# Dr Plant — Lemon V3 exploratory training

Run this notebook in Google Colab from an Android browser. It attempts to download the public Mendeley V3 archive through the documented public dataset API. If that endpoint requires OAuth or is unavailable, download the archive from the linked Mendeley page and place it in Google Drive at `MyDrive/DrPlant/8d9fv6kpt3-v3.zip`.

**Important:** this is an initial within-dataset exploratory baseline only. The published record does not yet provide per-image site/plant/session IDs, so the split is not independent. The results must not be represented as accuracy on unseen Syrian orchards.

In [ ]:
# Select Runtime > Change runtime type > GPU (T4 if available) before running.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path('/content/drplant-work')
DRIVE = Path('/content/drive/MyDrive/DrPlant')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
print('GPU:', __import__('torch').cuda.is_available())

In [ ]:
!git clone --depth 1 https://github.com/jonbander/dr-plant.git /content/dr-plant
%cd /content/dr-plant
!pip -q install Pillow scikit-learn

In [ ]:
# Try the documented public Mendeley Data API and archive endpoint.
import hashlib, json, requests, time
DATASET_ID = '8d9fv6kpt3'
VERSION = 3
ARCHIVE = DRIVE / '8d9fv6kpt3-v3.zip'
api_base = f'https://api.data.mendeley.com/datasets/{DATASET_ID}'
downloaded = False
try:
    meta_response = requests.get(api_base, params={'version': VERSION}, timeout=60)
    print('Metadata API HTTP:', meta_response.status_code)
    meta_response.raise_for_status()
    metadata = meta_response.json()
    print('Dataset:', metadata.get('name'), '| version:', metadata.get('version'))
    zip_response = requests.get(api_base + '/zip', params={'version': VERSION}, timeout=60)
    zip_response.raise_for_status()
    zip_meta = zip_response.json()
    print('Archive metadata:', {k: zip_meta.get(k) for k in ('status','size','sha256_hash')})
    for attempt in range(30):
        if zip_meta.get('status') in ('COMPLETE','COMPLETED','READY') or zip_meta.get('url'):
            break
        time.sleep(20)
        zip_response = requests.get(api_base + '/zip', params={'version': VERSION}, timeout=60)
        zip_response.raise_for_status(); zip_meta = zip_response.json()
    if not zip_meta.get('url') and zip_meta.get('status') not in ('COMPLETE','COMPLETED','READY'):
        raise RuntimeError('Mendeley archive is not ready through API yet.')
    file_response = requests.get(api_base + '/zip/file_downloaded', params={'version': VERSION}, stream=True, timeout=120)
    file_response.raise_for_status()
    h = hashlib.sha256()
    with ARCHIVE.open('wb') as out:
        for chunk in file_response.iter_content(8 * 1024 * 1024):
            if chunk: out.write(chunk); h.update(chunk)
    actual_hash = h.hexdigest()
    expected_hash = zip_meta.get('sha256_hash')
    if expected_hash and actual_hash.lower() != expected_hash.lower():
        ARCHIVE.unlink(missing_ok=True); raise RuntimeError('Archive SHA-256 mismatch; deleted the incomplete/untrusted file.')
    downloaded = True
    print('Downloaded archive:', ARCHIVE, '| bytes:', ARCHIVE.stat().st_size, '| SHA-256:', actual_hash)
except Exception as exc:
    print('Automatic API download unavailable:', repr(exc))
    print('Open https://data.mendeley.com/datasets/8d9fv6kpt3/3 and download the V3 archive.')
    print('Save it in Google Drive as:', ARCHIVE)
if not ARCHIVE.exists():
    raise FileNotFoundError(f'Put the downloaded V3 ZIP at {ARCHIVE}, then rerun this cell.')

In [ ]:
# Safely extract the archive and locate its class-folder root.
import zipfile
EXTRACTED = WORK / 'lemon-v3'
EXTRACTED.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    root = EXTRACTED.resolve()
    for info in z.infolist():
        target = (EXTRACTED / info.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError('Unsafe path inside ZIP: ' + info.filename)
    z.extractall(EXTRACTED)
image_ext = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
def is_class_root(path):
    dirs = [p for p in path.iterdir() if p.is_dir() and not p.is_symlink()]
    return len(dirs) >= 3 and sum(any(x.suffix.lower() in image_ext for x in d.rglob('*') if x.is_file()) for d in dirs) >= 3
candidates = [p for p in [EXTRACTED] + list(EXTRACTED.rglob('*')) if p.is_dir() and is_class_root(p)]
if not candidates: raise RuntimeError('Could not identify class folders. Inspect extracted archive layout before proceeding.')
IMAGE_ROOT = min(candidates, key=lambda p: len(p.parts))
print('Class root:', IMAGE_ROOT)
print('Class folders:', sorted(p.name for p in IMAGE_ROOT.iterdir() if p.is_dir()))

In [ ]:
# Inventory, SHA-256 duplicate scan, and image decoding check.
AUDIT = WORK / 'audit'
!python scripts/audit_image_classification_dataset.py "$IMAGE_ROOT" --dataset-id 8d9fv6kpt3 --version 3 --license 'CC BY 4.0 (Mendeley V3 record)' --out "$AUDIT" --verify-images
import json
audit_report = json.loads((AUDIT/'archive-audit-report.json').read_text())
print('Images:', audit_report['image_count'])
print('Class counts:', audit_report['class_image_counts'])
print('Exact duplicate groups:', audit_report['exact_duplicate_groups'])
print('Unreadable images:', len(audit_report['unreadable_images']))
if audit_report['image_count'] < 100: raise RuntimeError('Unexpectedly small archive; stop and inspect the dataset before training.')

In [ ]:
# Exploratory split: exact byte-identical files stay together; near-duplicates and site/plant/session leakage are NOT ruled out.
SPLITS = WORK / 'exploratory-splits'
!python scripts/prepare_exploratory_splits.py "$AUDIT/image-manifest.csv" --out "$SPLITS" --seed 42
print((SPLITS/'exploratory-split-report.json').read_text())

In [ ]:
# Train a first MobileNetV3-Small baseline. Test metrics are exploratory, not independent validation.
OUTPUT = WORK / 'model-output'
!python scripts/train_image_classifier.py "$IMAGE_ROOT" "$SPLITS/exploratory-splits.csv" --out "$OUTPUT" --epochs 12 --batch-size 32 --pretrained --exploratory-non-independent
print((OUTPUT/'metrics.json').read_text())

In [ ]:
# Preserve the model and metrics in Drive for later integration and review.
import shutil
DEST = DRIVE / 'lemon-v3-exploratory-baseline'
if DEST.exists(): shutil.rmtree(DEST)
shutil.copytree(OUTPUT, DEST)
shutil.copy2(AUDIT/'archive-audit-report.json', DEST/'archive-audit-report.json')
shutil.copy2(SPLITS/'exploratory-split-report.json', DEST/'exploratory-split-report.json')
print('Saved to Google Drive:', DEST)
print('Do not integrate this checkpoint into the app until an independent group-aware evaluation is available.')